In [ ]:
files = {
    "servidor/main.go": '''package main

import (
    "bufio"
    "fmt"
    "net"
    "os"
    "strings"
)

func handleConnection(conn net.Conn) {
    defer conn.Close()

    clientAddr := conn.RemoteAddr().String()
    fmt.Printf("[SERVIDOR] Novo cliente conectado: %s\n", clientAddr)

    scanner := bufio.NewScanner(conn)
    for scanner.Scan() {
        text := scanner.Text()
        fmt.Printf("[SERVIDOR] Recebido de %s: %s\n", clientAddr, text)

        if strings.EqualFold(strings.TrimSpace(text), "SAIR") {
            _, _ = conn.Write([]byte("Conexão encerrada pelo servidor. Tchau!"))
            break
        }

        _, err := conn.Write([]byte(text))
        if err != nil {
            fmt.Printf("[SERVIDOR] Erro ao responder %s: %v\n", clientAddr, err)
            break
        }
    }

    if err := scanner.Err(); err != nil {
        fmt.Printf("[SERVIDOR] Erro de leitura em %s: %v\n", clientAddr, err)
    }

    fmt.Printf("[SERVIDOR] Conexão finalizada com %s\n", clientAddr)
}

func main() {
    port := os.Getenv("ECHO_PORT")
    if port == "" {
        port = "5000"
    }

    address := ":" + port
    listener, err := net.Listen("tcp", address)
    if err != nil {
        fmt.Printf("[SERVIDOR] Erro ao iniciar o servidor: %v\n", err)
        os.Exit(1)
    }
    defer listener.Close()

    fmt.Printf("[SERVIDOR] Escutando na porta %s... Pronto para receber conexões.\n", port)

    for {
        conn, err := listener.Accept()
        if err != nil {
            fmt.Printf("[SERVIDOR] Erro ao aceitar conexão: %v\n", err)
            continue
        }
        go handleConnection(conn)
    }
}
''',

    "cliente/main.go": '''package main

import (
    "fmt"
    "net"
    "os"
    "time"
)

const (
    defaultHost               = "echo-server"
    defaultPort               = "5000"
    defaultMessage            = "Olá do cliente echo!"
    bufferSize                = 1024
    maxConnectionAttempts     = 15
    connectionTimeoutSeconds  = 5
)

func main() {
    host := os.Getenv("ECHO_HOST")
    if host == "" {
        host = defaultHost
    }

    port := os.Getenv("ECHO_PORT")
    if port == "" {
        port = defaultPort
    }

    message := os.Getenv("ECHO_MESSAGE")
    if message == "" {
        message = defaultMessage
    }

    address := net.JoinHostPort(host, port)
    timeout := time.Duration(connectionTimeoutSeconds) * time.Second

    for attempt := 1; attempt <= maxConnectionAttempts; attempt++ {
        conn, err := net.DialTimeout("tcp", address, timeout)
        if err != nil {
            fmt.Printf("[echo-client] Tentativa %d/%d falhou: %v. Aguardando servidor...\n", attempt, maxConnectionAttempts, err)
            time.Sleep(1 * time.Second)
            continue
        }

        fmt.Printf("[echo-client] Conectado em %s\n", address)
        _, err = conn.Write([]byte(message))
        if err != nil {
            conn.Close()
            fmt.Printf("[echo-client] Erro ao enviar mensagem: %v\n", err)
            os.Exit(1)
        }

        if tcpConn, ok := conn.(*net.TCPConn); ok {
            _ = tcpConn.CloseWrite()
        }

        responseBytes := make([]byte, 0, bufferSize)
        buffer := make([]byte, bufferSize)
        for {
            n, readErr := conn.Read(buffer)
            if n > 0 {
                responseBytes = append(responseBytes, buffer[:n]...)
            }
            if readErr != nil {
                break
            }
        }

        conn.Close()
        response := string(responseBytes)

        fmt.Printf("[echo-client] Enviado:  %s\n", message)
        fmt.Printf("[echo-client] Recebido: %s\n", response)

        if response != message {
            fmt.Printf("[echo-client] Resposta recebida difere da mensagem enviada. Esperado: %s. Recebido: %s.\n", message, response)
            os.Exit(1)
        }

        return
    }

    fmt.Fprintln(os.Stderr, "[echo-client] Não foi possível conectar ao servidor echo.")
    os.Exit(1)
}
''',

    "docker-compose.yml": '''version: '3.8'
services:
  echo-server:
    build: ./servidor
    container_name: echo-server
    ports:
      - "5000:5000"
    environment:
      - ECHO_PORT=5000

  echo-client:
    build: ./cliente
    container_name: echo-client
    depends_on:
      - echo-server
    environment:
      - ECHO_HOST=echo-server
      - ECHO_PORT=5000
      - ECHO_MESSAGE=Olá do cliente echo!
''',

    "servidor/Dockerfile": '''FROM golang:1.22-alpine
WORKDIR /app
COPY . .
RUN go build -o /usr/local/bin/echo-server .
CMD ["/usr/local/bin/echo-server"]
''',

    "cliente/Dockerfile": '''FROM golang:1.22-alpine
WORKDIR /app
COPY . .
RUN go build -o /usr/local/bin/echo-client .
CMD ["/usr/local/bin/echo-client"]
'''
}

for path, content in files.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

print("Arquivos gerados com sucesso.")


# Notebook para refatorar um cliente echo em Python para Go
Este notebook será usado para gerar os arquivos Go do cliente e as configurações Docker necessárias para alinhar o cliente com o servidor existente.